# 05 Evaluate all configs (Task 7: read-only, no GPU, no models)

Reads the result files written by notebook 04 (or `run.py` / `run_kaggle.py`) and builds every Task 7 output.

## How to use
1. Put every config's files in `results/`: `{config}_retrieval.json`, `{config}_answers.json`, `{config}_cost.json`.
   Teammates' zips (`results_<config>.zip` from notebook 04) can be dropped in `results/` too. The first code cell unzips them.
2. Edit `CONFIGS_TO_SCORE` if needed. A config with missing files is skipped with a warning.
3. Run all cells top to bottom.

## Outputs (all in `results/`)
| File | Content |
|---|---|
| `metrics.csv` | one row per config, the Task 7 columns (+ `manual_correct`, `manual_faithful` once filled) |
| `metrics_by_type.csv` | same metrics split by single-chunk / multi-chunk / unanswerable (and `qtype` if the test set has it) |
| `metrics_counts.csv` | raw counts such as 27/34, because 1 or 2 questions of difference is noise |
| `manual_check_{config}.csv` | 20 answerable questions (seed 42) for the team to fill with 1 or 0. Never overwritten once filled |
| `failures_{config}.csv` | no gold chunk in top 5, wrong citation, or wrong refusal |

## 1. Setup

In [ ]:
import os, sys, json, time, random, subprocess, zipfile
from pathlib import Path

REPO_URL = "https://github.com/yousefJooX/egyptian-personal-status-rag.git"
REPO_NAME = "egyptian-personal-status-rag"
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

cwd = Path.cwd()
if (cwd / "run.py").exists() or (cwd / "demo").exists():
    ROOT = cwd
else:
    ROOT = cwd / REPO_NAME
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)

DEMO = ROOT / "demo" if (ROOT / "demo").exists() else ROOT
os.chdir(DEMO)   # results/ lives here
for p in (str(DEMO), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)
print("Environment:", "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local")
print("ROOT:", ROOT, "| working dir:", Path.cwd())

## 2. Choose what to score

In [ ]:
# ============ EDIT THIS LIST ============
CONFIGS_TO_SCORE = ["arch1", "arch2", "arch3", "arch1_bge", "arch3_no_rr"]
# ========================================

SEED = 42

## 3. Load the test set and the result files

In [ ]:
import numpy as np
import pandas as pd

def find_file(name):
    for p in (DEMO / "data" / name, ROOT / "data" / name):
        if p.exists():
            return p
    raise FileNotFoundError(f"{name} not found in {DEMO/'data'} or {ROOT/'data'}")

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def mean(x):
    x = list(x)
    return float(np.mean(x)) if x else float("nan")

def qgroup(item):
    # question groups for the breakdown; also uses item["qtype"] if the test set has it
    if not item["answerable"]:
        g = {"size": "unanswerable"}
    else:
        g = {"size": "multi-chunk" if len(item["gold_chunks"]) > 1 else "single-chunk"}
    qt = item.get("qtype") or item.get("type")   # the test set labels kinds as "type"
    if qt:
        g["qtype"] = qt
    return g

results_dir = DEMO / "results"
results_dir.mkdir(exist_ok=True)

# unzip teammates' zips (skip their metrics.csv, never overwrite a manual_check file)
for z in sorted(results_dir.glob("*.zip")) + sorted(DEMO.glob("results_*.zip")):
    with zipfile.ZipFile(z) as zf:
        for name in zf.namelist():
            if Path(name).name != name or name == "metrics.csv":
                continue
            if name.startswith("manual_check_") and (results_dir / name).exists():
                continue
            zf.extract(name, results_dir)
        print("Unzipped", z.name)

test_set = load_json(find_file("test_set.json"))
T = {q["qid"]: q for q in test_set}
assert len(T) == len(test_set), "duplicate qid in test_set.json"

DATA = {}
for cfg in CONFIGS_TO_SCORE:
    fr, fa, fc = (results_dir / f"{cfg}_{s}.json" for s in ("retrieval", "answers", "cost"))
    missing = [f.name for f in (fr, fa) if not f.exists()]
    if missing:
        print(f"SKIPPED {cfg}: missing {missing}")
        continue
    ret, ans = load_json(fr), load_json(fa)
    if {r["qid"] for r in ret} != set(T) or {a["qid"] for a in ans} != set(T):
        print(f"SKIPPED {cfg}: question ids differ from test_set.json (re-run it with the current test set)")
        continue
    cost = load_json(fc) if fc.exists() else {}
    if not fc.exists():
        print(f"WARNING {cfg}: no cost file, peak_gpu_mb will be empty")
    DATA[cfg] = {"ret": {r["qid"]: [x["chunk_id"] for x in r["results"]][:5] for r in ret},
                 "ans": {a["qid"]: a for a in ans},
                 "peak": cost.get("peak_gpu_mb", float("nan"))}

assert DATA, "No config could be loaded. Check results/ and CONFIGS_TO_SCORE."
print("Scoring:", list(DATA))
print(f"Test set: {len(T)} questions, {sum(q['answerable'] for q in T.values())} answerable")

## 4. Metrics (definitions from `Task7-Evaluation.md`)
- **recall@5, mrr**: answerable questions only.
- **citation_acc**: answerable and not refused. Correct if `cited_chunks` is non-empty and every cited chunk is gold.
- **refusal_acc**: all questions. **avg_seconds**: all questions. **peak_gpu_mb**: from the cost file.

In [ ]:
def citation_ok(cfg, q):
    cited = DATA[cfg]["ans"][q]["cited_chunks"]
    return bool(cited) and set(cited) <= set(T[q]["gold_chunks"])

def evaluate(cfg, qids):
    RET, ANS = DATA[cfg]["ret"], DATA[cfg]["ans"]
    answerable = [q for q in qids if T[q]["answerable"]]
    recall, rr, found = [], [], 0
    for q in answerable:
        gold, top = set(T[q]["gold_chunks"]), RET[q]
        recall.append(len(gold & set(top)) / len(gold))
        ranks = [i for i, c in enumerate(top, 1) if c in gold]
        rr.append(1 / ranks[0] if ranks else 0.0)
        found += bool(gold & set(top))
    pool = [q for q in answerable if not ANS[q]["refused"]]
    cit = [citation_ok(cfg, q) for q in pool]
    ref = [ANS[q]["refused"] == (not T[q]["answerable"]) for q in qids]
    return {"recall@5": mean(recall), "mrr": mean(rr), "citation_acc": mean(cit),
            "refusal_acc": mean(ref), "avg_seconds": mean(ANS[q]["seconds"] for q in qids),
            "n": len(qids), "n_answerable": len(answerable), "found_top5": found,
            "n_cite_pool": len(pool), "n_cite_ok": sum(cit), "n_refusal_ok": sum(ref)}

COLS = ["config", "recall@5", "mrr", "citation_acc", "refusal_acc", "avg_seconds", "peak_gpu_mb"]
rows, counts = [], []
for cfg in DATA:
    m = evaluate(cfg, list(T))
    rows.append({"config": cfg, **{k: round(m[k], 3) for k in COLS[1:6]}, "peak_gpu_mb": DATA[cfg]["peak"]})
    counts.append({"config": cfg,
                   "recall_found_top5": f'{m["found_top5"]}/{m["n_answerable"]}',
                   "citation_correct": f'{m["n_cite_ok"]}/{m["n_cite_pool"]}',
                   "refusal_correct": f'{m["n_refusal_ok"]}/{m["n"]}'})

metrics = pd.DataFrame(rows, columns=COLS)
metrics["peak_gpu_mb"] = metrics["peak_gpu_mb"].astype("Int64")   # whole MB, empty if no cost file
metrics.to_csv(results_dir / "metrics.csv", index=False)
pd.DataFrame(counts).to_csv(results_dir / "metrics_counts.csv", index=False)
print(metrics.to_string(index=False))
print("\nCounts (a gap of 1 or 2 questions is noise):")
print(pd.DataFrame(counts).to_string(index=False))

## 5. Breakdown by question type

In [ ]:
rows = []
for cfg in DATA:
    groups = {}
    for q, t in T.items():
        for gname, gval in qgroup(t).items():
            groups.setdefault(f"{gname}: {gval}", []).append(q)
    for g, qids in groups.items():
        m = evaluate(cfg, qids)
        rows.append({"config": cfg, "group": g, "n": m["n"],
                     **{k: (round(m[k], 3) if m[k] == m[k] else None) for k in ("recall@5", "mrr", "citation_acc", "refusal_acc")}})
by_type = pd.DataFrame(rows)
by_type.to_csv(results_dir / "metrics_by_type.csv", index=False)
print(by_type.to_string(index=False))

## 6. Manual check and failures files
`manual_check_{config}.csv` is **not** overwritten if the team has already started filling it.

In [ ]:
def is_filled(path):
    try:
        m = pd.read_csv(path)
        return bool(m[["correct", "faithful"]].notna().any().any())
    except Exception:
        return False

answerable_ids = sorted(q for q in T if T[q]["answerable"])
picked = random.Random(SEED).sample(answerable_ids, min(20, len(answerable_ids)))

for cfg in DATA:
    RET, ANS = DATA[cfg]["ret"], DATA[cfg]["ans"]

    mpath = results_dir / f"manual_check_{cfg}.csv"
    if is_filled(mpath):
        print(f"{cfg}: manual_check already filled, kept as is")
    else:
        pd.DataFrame([{"qid": q, "question": T[q]["question"], "answer": ANS[q]["answer"],
                       "reference_answer": T[q]["reference_answer"], "correct": "", "faithful": ""}
                      for q in picked]).to_csv(mpath, index=False, encoding="utf-8-sig")

    fails = []
    for q, t in T.items():
        gold, a = set(t["gold_chunks"]), ANS[q]
        no_gold = t["answerable"] and not (gold & set(RET[q]))
        bad_cite = t["answerable"] and not a["refused"] and not citation_ok(cfg, q)
        bad_refusal = a["refused"] != (not t["answerable"])
        if no_gold or bad_cite or bad_refusal:
            fails.append({"qid": q, "question": t["question"], "gold_chunks": t["gold_chunks"],
                          "retrieved_top5": RET[q], "cited_chunks": a["cited_chunks"], "refused": a["refused"]})
    pd.DataFrame(fails, columns=["qid", "question", "gold_chunks", "retrieved_top5", "cited_chunks", "refused"]
                 ).to_csv(results_dir / f"failures_{cfg}.csv", index=False, encoding="utf-8-sig")
    print(f"{cfg}: {len(fails)} failures saved")

## 7. Hand check (required by Task 7)
Verify each metric by hand on 3 questions against the definitions. This prints the numbers to compare with your own.

In [ ]:
cfg0 = list(DATA)[0]
RET, ANS = DATA[cfg0]["ret"], DATA[cfg0]["ans"]
print("Config:", cfg0)
for q in random.Random(SEED).sample(answerable_ids, min(3, len(answerable_ids))):
    gold, top = set(T[q]["gold_chunks"]), RET[q]
    ranks = [i for i, c in enumerate(top, 1) if c in gold]
    print(f"{q} | gold: {sorted(gold)} | top5: {top}")
    print(f"   recall@5 = {len(gold & set(top)) / len(gold):.3f} | rr = {(1 / ranks[0] if ranks else 0):.3f}"
          f" | refused = {ANS[q]['refused']} | cited = {ANS[q]['cited_chunks']} | citation_ok = {citation_ok(cfg0, q)}")
un = [q for q in T if not T[q]["answerable"]][:1]
for q in un:
    print(f"{q} (unanswerable) | refused = {ANS[q]['refused']} -> refusal correct = {ANS[q]['refused']}")

## 8. After the team fills `manual_check_*.csv`
Fill `correct` and `faithful` with 1 or 0, put the files back in `results/`, and re-run this notebook (or just this cell after cell 4).
It adds `manual_correct` and `manual_faithful` (the means) to `metrics.csv`.

In [ ]:
mdf = pd.read_csv(results_dir / "metrics.csv")
for cfg in mdf["config"]:
    f = results_dir / f"manual_check_{cfg}.csv"
    if not f.exists():
        continue
    m = pd.read_csv(f)
    if len(m) and m["correct"].notna().all() and m["faithful"].notna().all():
        mdf.loc[mdf["config"] == cfg, "manual_correct"] = round(m["correct"].astype(float).mean(), 3)
        mdf.loc[mdf["config"] == cfg, "manual_faithful"] = round(m["faithful"].astype(float).mean(), 3)
    else:
        print(f"{cfg}: manual_check not fully filled yet, skipped")
mdf.to_csv(results_dir / "metrics.csv", index=False)
print(mdf.to_string(index=False))

## 9. Time split (optional, needs `{config}_timing.json` from notebook 04)

In [ ]:
rows = []
for cfg in DATA:
    f = results_dir / f"{cfg}_timing.json"
    if f.exists():
        rows.append({"config": cfg, **load_json(f)})
print(pd.DataFrame(rows).to_string(index=False) if rows else "No timing files found.")